In [ ]:
# JOB MARKET ANALYSIS — STEP 1: DATA CLEANING  (CORRECTED)
# Dataset: Data Analyst Job Postings (Luke Barousse / Kaggle)

import pandas as pd
import numpy as np
import ast

# LOAD DATA
df = pd.read_csv("gsearch_jobs.csv")
print("Shape (raw):", df.shape)
print("\nColumns:\n", df.columns.tolist())

# INITIAL INSPECTION
print("\n--- Missing Values ---")
print(df.isnull().sum())
print("\n--- Data Types ---")
print(df.dtypes)

# 3. DROP DUPLICATES
df.drop_duplicates(inplace=True)
print(f"\nAfter dropping duplicates: {df.shape}")

# 4. STANDARDISE COLUMN NAME
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

#CLEAN DATE COLUMN 
if "date_time" in df.columns:
    df["date_time"] = pd.to_datetime(df["date_time"], utc=True, errors="coerce")
    df["posting_date"]  = df["date_time"].dt.date
    # suppress timezone-drop warning — intentional
    df["posting_month"] = df["date_time"].dt.tz_localize(None).dt.to_period("M").astype(str)
    df["posting_year"]  = df["date_time"].dt.year
    print(f"\n✅ Date parsed. Date range: {df['posting_date'].min()} → {df['posting_date'].max()}")

# CLEAN SALARY COLUMNS
# FIX: this dataset uses 'salary_standardized' as the main salary column
salary_cols = ["salary_standardized", "salary_min", "salary_max", "salary_avg",
               "salary_hourly", "salary_yearly"]
for col in salary_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Filter salary_standardized to realistic annual salaries only
if "salary_standardized" in df.columns:
    before = df["salary_standardized"].notna().sum()
    df["salary_standardized"] = df["salary_standardized"].where(
        (df["salary_standardized"] >= 20_000) & (df["salary_standardized"] <= 400_000)
    )
    after = df["salary_standardized"].notna().sum()
    print(f"\n✅ salary_standardized: {before} → {after} valid rows (filtered outliers)")

# CLEAN JOB TITLE 
if "title" in df.columns:
    df["title"] = df["title"].str.strip().str.title()

# CLEAN LOCATION & EXTRACT STATE 
if "location" in df.columns:
    df["location"] = df["location"].str.strip()
    df["state"] = df["location"].str.extract(r",\s*([A-Z]{2})$")
    print(f"\n✅ State extracted. Top 5: {df['state'].value_counts().head().to_dict()}")

#  PARSE SKILLS (correct column: description_tokens) 
# FIX: column is 'description_tokens', NOT 'job_skills'
if "description_tokens" in df.columns:
    def safe_parse(val):
        if pd.isna(val):
            return []
        try:
            return ast.literal_eval(val)
        except Exception:
            return []

    df["skills_list"]  = df["description_tokens"].apply(safe_parse)
    df["skills_count"] = df["skills_list"].apply(len)
    jobs_with_skills   = df["skills_count"].gt(0).sum()
    print(f"\n✅ Skills parsed from 'description_tokens'")
    print(f"   Jobs with skills: {jobs_with_skills:,} / {len(df):,}")
else:
    print("⚠️  'description_tokens' column not found — check column names above")

# ── 10. HANDLE NULLS 
# Drop rows missing company name
if "company_name" in df.columns:
    before = len(df)
    df.dropna(subset=["company_name"], inplace=True)
    print(f"\n✅ Dropped {before - len(df)} rows with no company name")

# Fill schedule_type nulls
if "schedule_type" in df.columns:
    df["schedule_type"] = df["schedule_type"].fillna("Unknown")

# ── 11. CLEAN work_from_home COLUMN ──────────────────────────
# FIX: handle as boolean properly — no "Unknown" fill before mapping
if "work_from_home" in df.columns:
    df["work_from_home"] = (
        df["work_from_home"]
        .map({True: True, False: False, "True": True, "False": False, 1: True, 0: False})
        .fillna(False)   # NaN = not listed as remote = treat as False
        .astype(bool)
    )
    print(f"\n✅ work_from_home — Remote: {df['work_from_home'].sum():,} | On-site: {(~df['work_from_home']).sum():,}")

# ── 12. SAVE CLEANED DATA 
# Drop helper column (skills_list is a Python list; not needed in CSV)
df_save = df.drop(columns=["skills_list"], errors="ignore")
df_save.to_csv("gsearch_jobs_cleaned.csv", index=False)

print("\n" + "="*50)
print("✅ Cleaned data saved → gsearch_jobs_cleaned.csv")
print(f"   Final shape : {df_save.shape}")
print(f"   Columns     : {df_save.columns.tolist()}")

Shape (raw): (61953, 27)

Columns:
 ['Unnamed: 0', 'index', 'title', 'company_name', 'location', 'via', 'description', 'extensions', 'job_id', 'thumbnail', 'posted_at', 'schedule_type', 'work_from_home', 'salary', 'search_term', 'date_time', 'search_location', 'commute_time', 'salary_pay', 'salary_rate', 'salary_avg', 'salary_min', 'salary_max', 'salary_hourly', 'salary_yearly', 'salary_standardized', 'description_tokens']

--- Missing Values ---
Unnamed: 0                 0
index                      0
title                      0
company_name               0
location                  37
via                        9
description                0
extensions                 0
job_id                     0
thumbnail              23759
posted_at                190
schedule_type            246
work_from_home         33973
salary                 51865
search_term                0
date_time                  0
search_location            0
commute_time           61953
salary_pay             5186

In [2]:
df1 = pd.read_csv("gsearch_jobs_cleaned.csv")
print(df.shape)
print("\nSample of cleaned data:\n", df1.head(2))
print("\nShape of cleaned data:", df1.shape)

(61953, 33)

Sample of cleaned data:
    unnamed:_0  index         title company_name       location           via  \
0           0      0  Data Analyst         Meta       Anywhere  via LinkedIn   
1           1      1  Data Analyst          ATC  United States  via LinkedIn   

                                         description  \
0  In the intersection of compliance and analytic...   
1  Job Title: Entry Level Business Analyst / Prod...   

                                          extensions  \
0  ['15 hours ago', '101K–143K a year', 'Work fro...   
1  ['12 hours ago', 'Full-time', 'Health insurance']   

                                              job_id  \
0  eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QiLCJodGlkb2...   
1  eyJqb2JfdGl0bGUiOiJEYXRhIEFuYWx5c3QiLCJodGlkb2...   

                                           thumbnail  ... salary_max  \
0  https://encrypted-tbn0.gstatic.com/images?q=tb...  ...   143000.0   
1  https://encrypted-tbn0.gstatic.com/images?q=tb...  ...        NaN  